# E7 – E9: Feature Selection + Late Fusion Voting

**Owner: Bayo**

| Exp | Method | Input | Feature dim |
|-----|--------|-------|-------------|
| E7 | Feature Selection (top-200 RF) | Classical 252 + Deep 1280 → 1532-dim fused | 200 |
| E8 | Average Voting | E2 RF proba + E3 SVM proba (50/50) | separate |
| E9 | Weighted Voting | E2 RF proba + E3 SVM proba (grid-searched) | separate |

**Primary metric:** Macro-F1 (class imbalance up to 10:1)  
**Baselines:** E2 macro_f1=0.6476  |  E3 macro_f1=0.7848

In [ ]:
# ── Environment: Colab or local ──────────────────────────────────────────────
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = Path('/content/drive/MyDrive/CV/repo')
    FEAT_DIR  = Path('/content/drive/MyDrive/CV/05_features')
else:
    REPO_ROOT = next(
        (p for p in [Path.cwd()] + list(Path.cwd().parents) if (p / 'src').exists()),
        Path.cwd()
    )
    FEAT_DIR = REPO_ROOT / 'data/processed/features'

CLF_DIR     = REPO_ROOT / 'models/classifiers'
MDL_DIR     = REPO_ROOT / 'models'
FIG_DIR     = REPO_ROOT / 'figures/fusion'
RES_DIR     = REPO_ROOT / 'results'
METRICS_CSV = RES_DIR / 'metrics/classification_results.csv'
PRED_DIR    = RES_DIR / 'predictions'

FIG_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO_ROOT))
print('REPO_ROOT:', REPO_ROOT)
print('FEAT_DIR :', FEAT_DIR, '| exists:', FEAT_DIR.exists())
print('CLF_DIR  :', CLF_DIR,  '| exists:', CLF_DIR.exists())

In [ ]:
import json
import time
import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score

from src.utils.seed import set_seed, SEED
from src.evaluation.classification_metrics import (
    compute_all_metrics, save_metrics_row, get_confusion_matrix, CLASSES,
)
from src.evaluation.plots import plot_confusion_matrix
from src.fusion.late_fusion import average_voting, weighted_voting
from src.fusion.dimensionality_reduction import select_top_features

set_seed(SEED)
print('Classes:', CLASSES)

In [ ]:
# ── Load feature arrays ──────────────────────────────────────────────────────
# Use clean (no augmentation) train arrays — row-aligned across modalities
X_c_train = np.load(FEAT_DIR / 'classical_train_clean_X.npy')  # (45177, 252)
y_train    = np.load(FEAT_DIR / 'classical_train_clean_y.npy')
X_c_val   = np.load(FEAT_DIR / 'classical_val_X.npy')          # (9935, 252)
y_val      = np.load(FEAT_DIR / 'classical_val_y.npy')
X_c_test  = np.load(FEAT_DIR / 'classical_test_X.npy')         # (10553, 252)
y_test     = np.load(FEAT_DIR / 'classical_test_y.npy')

X_d_train = np.load(FEAT_DIR / 'deep_train_X.npy')             # (45177, 1280)
X_d_val   = np.load(FEAT_DIR / 'deep_val_X.npy')               # (9935, 1280)
X_d_test  = np.load(FEAT_DIR / 'deep_test_X.npy')              # (10553, 1280)

assert X_c_train.shape == (45177, 252)
assert X_d_train.shape == (45177, 1280)
np.testing.assert_array_equal(
    y_train, np.load(FEAT_DIR / 'deep_train_y.npy'),
    err_msg='classical/deep label mismatch on train'
)

print(f'Train {X_c_train.shape}  Val {X_c_val.shape}  Test {X_c_test.shape}')

---
## E7 · Feature Selection Fusion

Concatenate classical (252-dim) + deep (1280-dim) = **1532-dim** fused vector.  
A selector RF ranks all features; the top-200 are passed to a second (final) RF.

> **Two-RF rule:** selector RF = for importances only; final RF = saved model artifact.

In [ ]:
X_fused_train = np.concatenate([X_c_train, X_d_train], axis=1)  # (45177, 1532)
X_fused_val   = np.concatenate([X_c_val,   X_d_val  ], axis=1)
X_fused_test  = np.concatenate([X_c_test,  X_d_test ], axis=1)

FUSED_DIM = X_fused_train.shape[1]  # 1532
CLF_BOUNDARY = 252                   # classical ends here in the fused vector
print(f'Fused dim: {FUSED_DIM}  (classical 0-{CLF_BOUNDARY-1} | deep {CLF_BOUNDARY}-{FUSED_DIM-1})')

In [ ]:
TOP_K = 200

print(f'Selector RF — ranking {FUSED_DIM} features ...')
t0 = time.time()
importances, top_idx = select_top_features(X_fused_train, y_train, top_k=TOP_K)
selector_time_s = time.time() - t0
print(f'  Done in {selector_time_s:.1f}s')

np.save(MDL_DIR / 'e7_top200_feature_indices.npy', top_idx)

n_classical = int((top_idx < CLF_BOUNDARY).sum())
n_deep      = int((top_idx >= CLF_BOUNDARY).sum())
print(f'  Classical selected: {n_classical}/200 ({100*n_classical/TOP_K:.0f}%)')
print(f'  Deep selected:      {n_deep}/200 ({100*n_deep/TOP_K:.0f}%)')

X_sel_train = X_fused_train[:, top_idx]
X_sel_val   = X_fused_val[:,   top_idx]
X_sel_test  = X_fused_test[:,  top_idx]

In [ ]:
print('Final RF — training on 200 selected features ...')
t0 = time.time()
rf_e7 = RandomForestClassifier(
    n_estimators=200, class_weight='balanced', n_jobs=-1, random_state=SEED
)
rf_e7.fit(X_sel_train, y_train)
print(f'  Trained in {time.time()-t0:.1f}s')

e7_model_path = MDL_DIR / 'e7_feature_selection_rf.pkl'
joblib.dump(rf_e7, e7_model_path)
print(f'  Saved: {e7_model_path}  ({e7_model_path.stat().st_size/1e6:.1f} MB)')

In [ ]:
y_val_pred_e7  = rf_e7.predict(X_sel_val)
y_val_proba_e7 = rf_e7.predict_proba(X_sel_val)
val_f1_e7      = f1_score(y_val, y_val_pred_e7, average='macro', zero_division=0)

t0 = time.time()
y_pred_e7      = rf_e7.predict(X_sel_test)
y_proba_e7     = rf_e7.predict_proba(X_sel_test)
clf_time_s_e7  = time.time() - t0

print(f'E7  val  macro-F1 : {val_f1_e7:.4f}')
print(f'E7  test macro-F1 : {f1_score(y_test, y_pred_e7, average="macro", zero_division=0):.4f}')
print(f'E7  test accuracy : {accuracy_score(y_test, y_pred_e7):.4f}')

np.save(PRED_DIR / 'E7_predictions.npy',   y_pred_e7)
np.save(PRED_DIR / 'E7_probabilities.npy', y_proba_e7)

In [ ]:
# Feature group bar chart
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Classical\n(252-dim)', 'Deep\n(1280-dim)'],
              [n_classical, n_deep], color=['#4C72B0', '#DD8452'], edgecolor='white')
for bar, cnt in zip(bars, [n_classical, n_deep]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{cnt} ({100*cnt/TOP_K:.0f}%)', ha='center', va='bottom', fontsize=11)
ax.set_ylim(0, TOP_K + 30)
ax.set_ylabel('Features selected (out of 200)')
ax.set_title('E7 — Feature Group Breakdown')
plt.tight_layout()
plt.savefig(FIG_DIR / 'E7_feature_group_analysis.png', dpi=150)
plt.show()

# Confusion matrix
cm_e7 = get_confusion_matrix(y_test, y_pred_e7)
plot_confusion_matrix(cm_e7, 'E7 — Feature Selection (200-dim)',
                      str(FIG_DIR / 'E7_confusion_matrix.png'))

In [ ]:
row_e7 = compute_all_metrics(
    y_true=y_test, y_pred=y_pred_e7, y_prob=y_proba_e7,
    experiment_id='E7', model_name='RF_FeatureSelection_200',
    feature_dim=TOP_K,
    extraction_time_s=selector_time_s, n_train_samples=len(y_train),
    classification_time_s=clf_time_s_e7, n_test_samples=len(y_test),
    augmentation='none',
)
row_e7.update({
    'val_accuracy'           : round(float(accuracy_score(y_val, y_val_pred_e7)), 4),
    'val_macro_f1'           : round(float(val_f1_e7), 4),
    'feature_group_classical': n_classical,
    'feature_group_deep'     : n_deep,
    'timestamp'              : datetime.datetime.now().strftime('%Y-%m-%d %H:%M'),
})
save_metrics_row(row_e7, str(METRICS_CSV), 'E7')
print(f"E7 → accuracy={row_e7['accuracy']}  macro_f1={row_e7['macro_f1']}")

---
## E8 · Average Voting

Combine E2 RF (classical features) and E3 SVM (deep features) by averaging probabilities 50/50.  
No training — uses pre-trained models from Aly's v1.0 release.

> `svm_E3.pkl` = `{'clf': SVC(probability=True), 'scaler': StandardScaler}`  
> Apply `scaler.transform()` before calling `clf.predict_proba()`.

In [ ]:
rf_e2      = joblib.load(CLF_DIR / 'random_forest_E2.pkl')
svm_bundle = joblib.load(CLF_DIR / 'svm_E3.pkl')
svm_clf    = svm_bundle['clf']
svm_scaler = svm_bundle['scaler']

print('RF  E2  classes:', rf_e2.classes_)
print('SVM E3  probability:', svm_clf.probability)  # must be True

# Scale deep features for SVM
X_d_val_s  = svm_scaler.transform(X_d_val)
X_d_test_s = svm_scaler.transform(X_d_test)

In [ ]:
rf_proba_val   = rf_e2.predict_proba(X_c_val)     # (9935, 6)
rf_proba_test  = rf_e2.predict_proba(X_c_test)    # (10553, 6)
svm_proba_val  = svm_clf.predict_proba(X_d_val_s) # (9935, 6)
svm_proba_test = svm_clf.predict_proba(X_d_test_s)# (10553, 6)

print(f'RF  proba shape: {rf_proba_test.shape}   row-sum: {rf_proba_test[0].sum():.4f}')
print(f'SVM proba shape: {svm_proba_test.shape}  row-sum: {svm_proba_test[0].sum():.4f}')

In [ ]:
t0 = time.time()
avg_proba_test = average_voting(rf_proba_test, svm_proba_test)
y_pred_e8      = np.argmax(avg_proba_test, axis=1)
clf_time_s_e8  = time.time() - t0

avg_proba_val = average_voting(rf_proba_val, svm_proba_val)
y_val_pred_e8 = np.argmax(avg_proba_val, axis=1)
val_f1_e8     = f1_score(y_val, y_val_pred_e8, average='macro', zero_division=0)

print(f'E8  val  macro-F1 : {val_f1_e8:.4f}')
print(f'E8  test macro-F1 : {f1_score(y_test, y_pred_e8, average="macro", zero_division=0):.4f}')
print(f'E8  test accuracy : {accuracy_score(y_test, y_pred_e8):.4f}')

np.save(PRED_DIR / 'E8_predictions.npy',           y_pred_e8)
np.save(PRED_DIR / 'E8_probabilities.npy',         avg_proba_test)
np.save(PRED_DIR / 'E8_rf_probabilities_test.npy',  rf_proba_test)
np.save(PRED_DIR / 'E8_svm_probabilities_test.npy', svm_proba_test)

In [ ]:
cm_e8 = get_confusion_matrix(y_test, y_pred_e8)
plot_confusion_matrix(cm_e8, 'E8 — Average Voting (50/50)', str(FIG_DIR / 'E8_confusion_matrix.png'))

row_e8 = compute_all_metrics(
    y_true=y_test, y_pred=y_pred_e8, y_prob=avg_proba_test,
    experiment_id='E8', model_name='AverageVoting_RF+SVM',
    feature_dim=252 + 1280,
    extraction_time_s=0.0, n_train_samples=len(y_train),
    classification_time_s=clf_time_s_e8, n_test_samples=len(y_test),
    augmentation='none',
)
row_e8.update({
    'val_accuracy' : round(float(accuracy_score(y_val, y_val_pred_e8)), 4),
    'val_macro_f1' : round(float(val_f1_e8), 4),
    'timestamp'    : datetime.datetime.now().strftime('%Y-%m-%d %H:%M'),
})
save_metrics_row(row_e8, str(METRICS_CSV), 'E8')
print(f"E8 → accuracy={row_e8['accuracy']}  macro_f1={row_e8['macro_f1']}")

---
## E9 · Weighted Voting

Grid search `w_rf` ∈ [0.05, 0.95] in 19 steps, maximising **macro-F1 on the validation set**.  
Test set used exactly once after best weight is chosen.

In [ ]:
STEPS   = 19
weights = np.linspace(0.05, 0.95, STEPS)
records = []
best_f1, best_w_rf = -1.0, 0.5

for w_rf in weights:
    w_svm = 1.0 - w_rf
    fused = weighted_voting(rf_proba_val, svm_proba_val, weight_a=w_rf)
    preds = np.argmax(fused, axis=1)
    acc   = float(accuracy_score(y_val, preds))
    mf1   = float(f1_score(y_val, preds, average='macro', zero_division=0))
    records.append({'w_rf': round(w_rf, 4), 'w_svm': round(w_svm, 4),
                    'val_accuracy': round(acc, 4), 'val_macro_f1': round(mf1, 4)})
    if mf1 > best_f1:
        best_f1, best_w_rf = mf1, w_rf

best_w_svm = 1.0 - best_w_rf
df_ws = pd.DataFrame(records)
print(f'Best  w_rf={best_w_rf:.4f}  w_svm={best_w_svm:.4f}  val_macro_f1={best_f1:.4f}')
print(df_ws.to_string(index=False))

In [ ]:
ws_csv = RES_DIR / 'e9_weight_search_results.csv'
df_ws.to_csv(ws_csv, index=False)

with open(RES_DIR / 'e9_optimal_weights.json', 'w') as f:
    json.dump({'w_rf': round(best_w_rf, 4), 'w_svm': round(best_w_svm, 4),
               'val_macro_f1': round(best_f1, 4)}, f, indent=2)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df_ws['w_rf'], df_ws['val_macro_f1'], marker='o', label='macro-F1')
ax.plot(df_ws['w_rf'], df_ws['val_accuracy'], marker='s', ls='--', label='accuracy')
ax.axvline(best_w_rf, color='red', ls=':', lw=1.5, label=f'best w_rf={best_w_rf:.2f}')
ax.set_xlabel('w_rf (RF weight)   [w_svm = 1 – w_rf]')
ax.set_ylabel('Validation score')
ax.set_title('E9 — Weight Grid Search on Validation Set')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'E9_weight_search.png', dpi=150)
plt.show()
print('Weight search saved.')

In [ ]:
# ── Apply best weights to TEST SET (exactly once) ────────────────────────────
t0 = time.time()
w_proba_test  = weighted_voting(rf_proba_test, svm_proba_test, weight_a=best_w_rf)
y_pred_e9     = np.argmax(w_proba_test, axis=1)
clf_time_s_e9 = time.time() - t0

print(f'E9  test macro-F1 : {f1_score(y_test, y_pred_e9, average="macro", zero_division=0):.4f}')
print(f'E9  test accuracy : {accuracy_score(y_test, y_pred_e9):.4f}')

np.save(PRED_DIR / 'E9_predictions.npy',   y_pred_e9)
np.save(PRED_DIR / 'E9_probabilities.npy', w_proba_test)

In [ ]:
cm_e9 = get_confusion_matrix(y_test, y_pred_e9)
plot_confusion_matrix(
    cm_e9, f'E9 — Weighted Voting (w_rf={best_w_rf:.2f})',
    str(FIG_DIR / 'E9_confusion_matrix.png')
)

row_e9 = compute_all_metrics(
    y_true=y_test, y_pred=y_pred_e9, y_prob=w_proba_test,
    experiment_id='E9', model_name=f'WeightedVoting_RF+SVM_wrf{best_w_rf:.2f}',
    feature_dim=252 + 1280,
    extraction_time_s=0.0, n_train_samples=len(y_train),
    classification_time_s=clf_time_s_e9, n_test_samples=len(y_test),
    augmentation='none',
)
row_e9.update({
    'val_macro_f1' : round(float(best_f1), 4),
    'optimal_w_rf' : round(float(best_w_rf), 4),
    'optimal_w_svm': round(float(best_w_svm), 4),
    'timestamp'    : datetime.datetime.now().strftime('%Y-%m-%d %H:%M'),
})
save_metrics_row(row_e9, str(METRICS_CSV), 'E9')
print(f"E9 → accuracy={row_e9['accuracy']}  macro_f1={row_e9['macro_f1']}")

---
## Summary — E7/E8/E9 vs Baselines

In [ ]:
df_all = pd.read_csv(METRICS_CSV)
cols   = ['experiment', 'model', 'feature_dim', 'accuracy', 'macro_f1',
          'balanced_accuracy', 'inference_ms_per_crop']
show   = df_all[df_all['experiment'].isin(['E2', 'E3', 'E7', 'E8', 'E9'])][cols]
print(show.sort_values('macro_f1', ascending=False).to_string(index=False))